# 02 Cleaning and Splitting

## Purpose

This notebook takes the corrected interim dataset from `01_raw_data_audit`
and creates the clean base dataset for supervised binary classification.

The goal of this stage is deliberately narrow:

- load the corrected interim dataset;
- create the binary modelling target;
- define the modelling feature set;
- exclude identifiers and non-modelling target columns;
- define semantic feature groups;
- validate that all modelling features are assigned to exactly one group;
- create a stratified train-test split;
- save the training and test datasets.

This file does **not** perform exploratory feature-target analysis, one-hot
encoding, scaling, imputation, PCA, feature selection, class-imbalance
resampling, model training, or threshold tuning.


## Why the split happens before EDA and modelling

The held-out test set is meant to approximate future unseen data. Therefore,
it should not influence feature engineering, preprocessing choices, model
selection, hyperparameter tuning, or threshold selection.

After this notebook creates `train.csv` and `test.csv`, all target-based EDA
and all modelling decisions should use only the training set. The test set is
kept aside until final evaluation.

The only steps performed before splitting are deterministic data-quality and
dataset-construction steps. These are necessary to build a valid modelling
table and do not use feature-target patterns or model performance.


In [1]:
from pathlib import Path

import pandas as pd

from sklearn.model_selection import train_test_split


In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.4f}".format)


## Configuration

The project root is detected by searching upward from the current working
directory. This makes the notebook executable from either the repository root
or the `notebooks/` folder.


In [3]:
def find_project_root(start: Path | None = None) -> Path:
    """Return the project root by searching upward for project marker files.

    Parameters
    ----------
    start:
        Optional starting directory. If omitted, the current working directory is
        used.

    Returns
    -------
    Path
        Repository root directory.

    Raises
    ------
    FileNotFoundError
        If no plausible project root can be found.
    """
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        has_repo_markers = (
            (candidate / "pyproject.toml").exists()
            or (candidate / "README.md").exists()
        )
        has_project_dirs = (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "reports").exists()
        )

        if has_repo_markers and has_project_dirs:
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside the "
        "repository, or add project marker files such as pyproject.toml."
    )


In [4]:
PROJECT_ROOT = find_project_root()

INTERIM_DATA_PATH = PROJECT_ROOT / "data" / "interim" / "telco_churn_interim.csv"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DATA_PATH = PROCESSED_DATA_DIR / "train.csv"
TEST_DATA_PATH = PROCESSED_DATA_DIR / "test.csv"

TARGET_COLUMN = "Churn"
BINARY_TARGET_COLUMN = "Churn_binary"
POSITIVE_CLASS = "Yes"
NEGATIVE_CLASS = "No"

IDENTIFIER_COLUMN = "customerID"

RANDOM_STATE = 42
TEST_SIZE = 0.20


In [5]:
path_check = pd.DataFrame(
    {
        "item": [
            "current_working_directory",
            "project_root",
            "interim_data_path",
            "interim_data_path_exists",
            "processed_data_dir",
            "train_data_path",
            "test_data_path",
        ],
        "value": [
            str(Path.cwd()),
            str(PROJECT_ROOT),
            str(INTERIM_DATA_PATH),
            INTERIM_DATA_PATH.exists(),
            str(PROCESSED_DATA_DIR),
            str(TRAIN_DATA_PATH),
            str(TEST_DATA_PATH),
        ],
    }
)

path_check


,item,value
0,current_working_directory,C:\Projects_Data\classification\telco-customer...
1,project_root,C:\Projects_Data\classification\telco-customer...
2,interim_data_path,C:\Projects_Data\classification\telco-customer...
3,interim_data_path_exists,True
4,processed_data_dir,C:\Projects_Data\classification\telco-customer...
5,train_data_path,C:\Projects_Data\classification\telco-customer...
6,test_data_path,C:\Projects_Data\classification\telco-customer...


## Load interim dataset

The interim dataset is produced by `01_raw_data_audit`. It preserves the raw
columns but corrects the representation of `TotalCharges`.


In [6]:
if not INTERIM_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Interim dataset not found at: {INTERIM_DATA_PATH}\n"
        "Run 01_raw_data_audit.py or 01_raw_data_audit.ipynb first."
    )

df_interim = pd.read_csv(INTERIM_DATA_PATH)


In [7]:
df_interim.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.8500,29.8500,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.9500,"1,889.5000",No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.8500,108.1500,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3000,"1,840.7500",No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7000,151.6500,Yes


In [8]:
interim_overview = pd.DataFrame(
    {
        "item": [
            "number_of_rows",
            "number_of_columns",
            "duplicate_rows",
            "total_missing_values",
            "columns_with_missing_values",
        ],
        "value": [
            df_interim.shape[0],
            df_interim.shape[1],
            int(df_interim.duplicated().sum()),
            int(df_interim.isna().sum().sum()),
            int((df_interim.isna().sum() > 0).sum()),
        ],
    }
)

interim_overview


,item,value
0,number_of_rows,7043
1,number_of_columns,21
2,duplicate_rows,0
3,total_missing_values,0
4,columns_with_missing_values,0


In [9]:
interim_schema = pd.DataFrame(
    {
        "column": df_interim.columns,
        "dtype": df_interim.dtypes.astype(str).values,
        "count": len(df_interim),
        "missing_count": df_interim.isna().sum().values,
        "missing_percentage": 100 * df_interim.isna().mean().values,
        "unique_values": df_interim.nunique(dropna=False).values,
    }
)

interim_schema


,column,dtype,count,missing_count,missing_percentage,unique_values
0,customerID,str,7043,0,0.0000,7043
1,gender,str,7043,0,0.0000,2
2,SeniorCitizen,int64,7043,0,0.0000,2
3,Partner,str,7043,0,0.0000,2
4,Dependents,str,7043,0,0.0000,2
5,tenure,int64,7043,0,0.0000,73
6,PhoneService,str,7043,0,0.0000,2
7,MultipleLines,str,7043,0,0.0000,3
8,InternetService,str,7043,0,0.0000,3
9,OnlineSecurity,str,7043,0,0.0000,3


## Create binary target

The supervised learning target is `Churn_binary`.

The positive class is churn:

$$
\texttt{Churn = Yes} \rightarrow \texttt{Churn\_binary = 1}.
$$

The negative class is non-churn:

$$
\texttt{Churn = No} \rightarrow \texttt{Churn\_binary = 0}.
$$


In [10]:
df_clean = df_interim.copy()

target_mapping = {
    NEGATIVE_CLASS: 0,
    POSITIVE_CLASS: 1,
}

df_clean[BINARY_TARGET_COLUMN] = df_clean[TARGET_COLUMN].map(target_mapping)

target_encoding_check = (
    df_clean[[TARGET_COLUMN, BINARY_TARGET_COLUMN]]
    .drop_duplicates()
    .sort_values(BINARY_TARGET_COLUMN)
    .reset_index(drop=True)
)

target_encoding_check


,Churn,Churn_binary
0,No,0
1,Yes,1


In [11]:
target_missing_after_encoding = int(df_clean[BINARY_TARGET_COLUMN].isna().sum())

target_encoding_summary = pd.DataFrame(
    {
        "item": [
            "missing_values_after_binary_encoding",
            "observed_binary_target_values",
        ],
        "value": [
            target_missing_after_encoding,
            sorted(df_clean[BINARY_TARGET_COLUMN].dropna().unique().tolist()),
        ],
    }
)

target_encoding_summary


,item,value
0,missing_values_after_binary_encoding,0
1,observed_binary_target_values,"[0, 1]"


In [12]:
if target_missing_after_encoding != 0:
    raise ValueError(
        "The binary target contains missing values after encoding. Check the "
        "raw target labels before splitting."
    )


## Define modelling feature set

`customerID` is excluded because it is a unique identifier. The original
string target `Churn` is excluded because the model target is the numeric
binary column `Churn_binary`.

The resulting modelling table contains 19 feature columns and one target
column.


In [13]:
modelling_excluded_columns = [
    IDENTIFIER_COLUMN,
    TARGET_COLUMN,
    BINARY_TARGET_COLUMN,
]

feature_columns = [
    column
    for column in df_clean.columns
    if column not in modelling_excluded_columns
]

feature_columns


['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges']

In [14]:
df_model = df_clean[feature_columns + [BINARY_TARGET_COLUMN]].copy()

# A separate traceability table is kept in memory. It is not used for modelling.
id_trace = df_clean[[IDENTIFIER_COLUMN]].copy()

model_table_overview = pd.DataFrame(
    {
        "item": [
            "model_table_rows",
            "model_table_columns",
            "number_of_feature_columns",
            "target_column",
            "identifier_excluded_from_features",
            "original_string_target_excluded_from_features",
        ],
        "value": [
            df_model.shape[0],
            df_model.shape[1],
            len(feature_columns),
            BINARY_TARGET_COLUMN,
            IDENTIFIER_COLUMN not in feature_columns,
            TARGET_COLUMN not in feature_columns,
        ],
    }
)

model_table_overview


,item,value
0,model_table_rows,7043
1,model_table_columns,20
2,number_of_feature_columns,19
3,target_column,Churn_binary
4,identifier_excluded_from_features,True
5,original_string_target_excluded_from_features,True


In [15]:
df_model.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn_binary
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.8500,29.8500,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.9500,"1,889.5000",0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.8500,108.1500,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3000,"1,840.7500",0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7000,151.6500,1


## Define semantic feature groups

These groups describe the clean base dataset. They do not yet determine the
final preprocessing for every model.

Later model-specific pipelines can decide whether to one-hot encode, scale,
pass through, or otherwise transform these features. For example:

- linear models, kNN, SVMs, and neural networks usually require scaling after
  encoding;
- tree-based models need categorical encoding in scikit-learn but do not
  require scaling;
- Naive Bayes variants may require different representations depending on the
  assumed feature distribution.


In [16]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

binary_categorical_features = [
    "SeniorCitizen",
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]

nominal_categorical_features = [
    column
    for column in feature_columns
    if column not in numeric_features + binary_categorical_features
]

feature_group_table = pd.DataFrame(
    {
        "feature_group": [
            "numeric_features",
            "binary_categorical_features",
            "nominal_categorical_features",
        ],
        "count": [
            len(numeric_features),
            len(binary_categorical_features),
            len(nominal_categorical_features),
        ],
        "features": [
            numeric_features,
            binary_categorical_features,
            nominal_categorical_features,
        ],
    }
)

feature_group_table


,feature_group,count,features
0,numeric_features,3,"[tenure, MonthlyCharges, TotalCharges]"
1,binary_categorical_features,6,"[SeniorCitizen, gender, Partner, Dependents, P..."
2,nominal_categorical_features,10,"[MultipleLines, InternetService, OnlineSecurit..."


In [17]:
all_grouped_features = (
    numeric_features
    + binary_categorical_features
    + nominal_categorical_features
)

feature_group_validation = pd.DataFrame(
    {
        "check": [
            "number_of_feature_columns",
            "number_of_grouped_features",
            "features_missing_from_groups",
            "grouped_features_not_in_feature_columns",
            "duplicate_grouped_features",
        ],
        "value": [
            len(feature_columns),
            len(all_grouped_features),
            sorted(set(feature_columns) - set(all_grouped_features)),
            sorted(set(all_grouped_features) - set(feature_columns)),
            sorted(
                [
                    feature
                    for feature in set(all_grouped_features)
                    if all_grouped_features.count(feature) > 1
                ]
            ),
        ],
    }
)

feature_group_validation


,check,value
0,number_of_feature_columns,19
1,number_of_grouped_features,19
2,features_missing_from_groups,[]
3,grouped_features_not_in_feature_columns,[]
4,duplicate_grouped_features,[]


In [18]:
if len(feature_columns) != len(all_grouped_features):
    raise ValueError("Feature group validation failed: feature counts do not match.")

if set(feature_columns) != set(all_grouped_features):
    raise ValueError("Feature group validation failed: grouped features do not match.")

if len(all_grouped_features) != len(set(all_grouped_features)):
    raise ValueError("Feature group validation failed: duplicate grouped features found.")


## Final checks before splitting

The final modelling table is checked for missing values and duplicate rows.

Duplicate feature-target combinations after removing `customerID` are not
automatically a data-quality problem. Multiple customers can genuinely share
the same observed characteristics and churn label.


In [19]:
missing_summary_model = (
    df_model
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary_model["missing_percentage"] = (
    100 * missing_summary_model["missing_count"] / len(df_model)
)

missing_summary_model = missing_summary_model.sort_values(
    "missing_count",
    ascending=False,
).reset_index(drop=True)

missing_summary_model


,column,missing_count,missing_percentage
0,gender,0,0.0000
1,SeniorCitizen,0,0.0000
2,Partner,0,0.0000
3,Dependents,0,0.0000
4,tenure,0,0.0000
5,PhoneService,0,0.0000
6,MultipleLines,0,0.0000
7,InternetService,0,0.0000
8,OnlineSecurity,0,0.0000
9,OnlineBackup,0,0.0000


In [20]:
duplicate_summary = pd.DataFrame(
    {
        "item": [
            "duplicates_full_interim_data",
            "duplicates_model_table_feature_target_rows",
        ],
        "count": [
            int(df_interim.duplicated().sum()),
            int(df_model.duplicated().sum()),
        ],
    }
)

duplicate_summary


,item,count
0,duplicates_full_interim_data,0
1,duplicates_model_table_feature_target_rows,22


## Target distribution before splitting

The target distribution is checked to justify stratified splitting. Since the
positive class is a minority class, stratification helps preserve the churn
proportion in both train and test sets.


In [21]:
target_distribution_clean = (
    df_model[BINARY_TARGET_COLUMN]
    .value_counts(dropna=False)
    .rename_axis(BINARY_TARGET_COLUMN)
    .reset_index(name="count")
)

target_distribution_clean["percentage"] = (
    100
    * target_distribution_clean["count"]
    / target_distribution_clean["count"].sum()
)

target_distribution_clean


,Churn_binary,count,percentage
0,0,5174,73.4630
1,1,1869,26.5370


## Stratified train-test split

The test set is held out for final evaluation. After this point, target-based
EDA, feature engineering, preprocessing choices, model selection, and
hyperparameter tuning should use only the training set.

The split uses:

- `test_size = 0.20`;
- `random_state = 42`;
- stratification by `Churn_binary`.


In [22]:
train_df, test_df = train_test_split(
    df_model,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_model[BINARY_TARGET_COLUMN],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_overview = pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(train_df), len(test_df)],
        "percentage": [
            100 * len(train_df) / len(df_model),
            100 * len(test_df) / len(df_model),
        ],
    }
)

split_overview


,split,rows,percentage
0,train,5634,79.9943
1,test,1409,20.0057


In [23]:
def target_distribution_by_split(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    target_column: str,
) -> pd.DataFrame:
    """Return target counts and percentages by split."""
    output_tables = []

    for split_name, split_data in [
        ("train", train_data),
        ("test", test_data),
    ]:
        table = (
            split_data[target_column]
            .value_counts(dropna=False)
            .rename_axis(target_column)
            .reset_index(name="count")
        )

        table.insert(0, "split", split_name)
        table["percentage"] = 100 * table["count"] / table["count"].sum()

        output_tables.append(table)

    return pd.concat(output_tables, ignore_index=True)


split_target_distribution = target_distribution_by_split(
    train_df,
    test_df,
    BINARY_TARGET_COLUMN,
)

split_target_distribution


,split,Churn_binary,count,percentage
0,train,0,4139,73.4647
1,train,1,1495,26.5353
2,test,0,1035,73.4564
3,test,1,374,26.5436


## Save processed train and test datasets

The processed train and test files are saved locally. They should usually not
be committed to Git if the project keeps data files out of version control.

The next workflow step should load only `train.csv` for training-set EDA.


In [24]:
train_df.to_csv(TRAIN_DATA_PATH, index=False)
test_df.to_csv(TEST_DATA_PATH, index=False)

save_check = pd.DataFrame(
    {
        "item": [
            "train_data_path",
            "test_data_path",
            "train_data_path_exists",
            "test_data_path_exists",
            "train_rows",
            "test_rows",
            "train_missing_values",
            "test_missing_values",
        ],
        "value": [
            str(TRAIN_DATA_PATH),
            str(TEST_DATA_PATH),
            TRAIN_DATA_PATH.exists(),
            TEST_DATA_PATH.exists(),
            len(train_df),
            len(test_df),
            int(train_df.isna().sum().sum()),
            int(test_df.isna().sum().sum()),
        ],
    }
)

save_check


,item,value
0,train_data_path,C:\Projects_Data\classification\telco-customer...
1,test_data_path,C:\Projects_Data\classification\telco-customer...
2,train_data_path_exists,True
3,test_data_path_exists,True
4,train_rows,5634
5,test_rows,1409
6,train_missing_values,0
7,test_missing_values,0
